# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset via Croissant
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (fields as attributes, not via subscripting!)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we enumerate all available record sets in the dataset, list their `@id`s, their labels, and preview their available fields.

In [ ]:
# List all record sets in the dataset
record_sets = dataset.metadata.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, label: {rs.get('name', rs.get('@id'))}")

print("\nFields within each record set:")
for rs in record_sets:
    print(f"\nRecord set: {rs['@id']}")
    fields = rs.get('fields', [])
    for f in fields:
        print(f"  - Field @id: {f['@id']}, name: {f.get('name', f['@id'])}, dataType: {f.get('dataType', 'unknown')}")


## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use record set and field `@id`s from the overview above.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        print(f"Loaded: {len(df)} records, columns: {df.columns.tolist()}")
        dataframes[record_set_id] = df
    else:
        print("No records retrieved.")
if dataframes:
    # Pick first record set as example for further analysis
    example_record_set_id = list(dataframes.keys())[0]
    print(f"\nUsing record set for further analysis: {example_record_set_id}")
    print("Columns:", dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print("No tabular record sets loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on criteria, normalizing numeric fields, and grouping by key attributes.
Here we demonstrate filtering and normalization using a numeric field.

In [ ]:
# Choose one record set for EDA:
record_set_id = example_record_set_id
df = dataframes[record_set_id]
print(f"EDA on record set: {record_set_id}")

# Identify a numeric field (by type 'Float' or 'Integer'); choose the first one found
numeric_field_id = None
for rs in dataset.metadata.record_sets:
    if rs['@id'] == record_set_id:
        for field in rs.get('fields', []):
            if field.get('dataType') in ['Float', 'Integer', 'Number']:
                numeric_field_id = field['@id']
                print(f"Chosen numeric field: {numeric_field_id}")
                break
        break
if numeric_field_id is None:
    print("No numeric field found. Skipping EDA.")
else:
    # Filter: e.g., keep only records where numeric_field_id > threshold
    threshold = df[numeric_field_id].mean() if np.issubdtype(df[numeric_field_id].dtype, np.number) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Identify a categorical/group-by field (first non-numeric)
    group_field = None
    for rs in dataset.metadata.record_sets:
        if rs['@id'] == record_set_id:
            for field in rs.get('fields', []):
                if field['@id'] != numeric_field_id and field.get('dataType') not in ['Float', 'Integer', 'Number']:
                    if field['@id'] in filtered_df.columns:
                        group_field = field['@id']
                        print(f"Using group field: {group_field}")
                        break
            break

    if group_field:
        grouped = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"\nGrouped by {group_field}, mean of {numeric_field_id}:")
        display(grouped.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    # Simple histogram of the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # If group_field available, boxplot by group
    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² clinical colorectal cancer dataset using the `mlcroissant` library.

- We identified available record sets and fields by their `@id`s,
- Extracted tabular data from record sets for inspection and processing,
- Applied simple exploratory analysis, such as filtering, normalization, grouping, and visualization of field distributions.

Refer to the Croissant schema and field `@id`s when performing more advanced analysis or linking additional resources.